# Apply High-Frequency SNP Variants to Reference Genome (VCF → FASTA)

This notebook:
- Loads a multi-allelic VCF file with allele frequencies
- Loads a reference genome in FASTA format
- Applies the most common allele (based on frequency) at each SNP position
- Writes a new FASTA with updated sequences
- Logs which variants were applied

Tools Used: scikit-allel, Biopython


In [ ]:
# scikit-allel: specialist library for reading and manipulating genomic variant files (VCF format)
# handles the complexity of multiallelic VCF parsing that would take weeks to implement manually
import allel

# numpy: numerical computing library
# not called directly in this script but used implicitly by scikit-allel for array operations
import numpy as np

# SeqIO: Biopython's input/output handler - reads and writes biological sequence files (FASTA, GenBank etc.)
# used to both parse the reference FASTA and write the final consensus FASTA output
from Bio import SeqIO

# Seq: represents a biological sequence as a string of nucleotide characters (A, T, G, C)
# used at the output stage to reconstruct the modified sequence before writing
from Bio.Seq import Seq

# SeqRecord: a container that bundles a Seq object together with its metadata (ID, description)
# required by SeqIO.write - each FASTA entry needs both a sequence and an identifier
from Bio.SeqRecord import SeqRecord

## Step 1: Load the VCF file

VCF = Variant Call Format  
read specific fields: chromosome, position, REF allele, ALT alleles, and allele frequency (AF).

In [ ]:

# setting path to vcf
vcf_path = "path/to/g31016_tetraploids_multiallelic.vcf" 

# reading in the vcf file using scikit-allel
# extracting the following
# chrom - info on the chromosome 
# pos - pos of the variants 
# ref - reference allele seq
# alt - alternative allele seq
# AF - allele freq value 

# allel.read_vcf returns a dictionary (callset) where each key is a field name
# and each value is a numpy array of that field's values across all variants
# fields is a list of strings specifying exactly which fields to extract from the VCF
# only extracting what we need rather than the entire VCF - more efficient, less memory usage
# types is a dictionary associating specific fields with their required data types
# types={'variants/AF': 'f8'} explicitly tells scikit-allel to read allele frequency as a 64-bit float
# without this AF might be read as a string, breaking the mathematical comparisons in the filtering logic later
# f8 is a numpy dtype meaning 64-bit floating point - gives precision when comparing frequency values
# other fields (CHROM, POS, REF, ALT) don't need explicit types as scikit-allel handles strings and integers correctly by default
callset = allel.read_vcf(
    vcf_path,
    fields=['variants/CHROM', 'variants/POS', 'variants/REF', 'variants/ALT', 'variants/AF'],
    types={'variants/AF': 'f8'} # make sure allele freq is recognised as a float 
)

# extracting individual data arrays from the callset dictionary
# unpacking into named variables makes downstream code more readable
# each variable is a numpy array with one entry per variant in the VCF
positions = callset['variants/POS']      # absolute genomic position of each variant
ref_alleles = callset['variants/REF']    # reference allele at each position
alt_alleles = callset['variants/ALT']    # alternative allele(s) at each position - may be multiple per variant
chroms = callset['variants/CHROM']       # chromosome each variant belongs to
allele_freqs = callset['variants/AF']    # frequency of each allele in the population (0.0 - 1.0)
variant_count = len(positions)           # counting variants - used to drive the loop in Step 3

print("Loaded VCF with {} variants.".format(variant_count))

## Step 2: Load the reference FASTA file

We store sequences as lists of characters so they can be modified in place.


In [ ]:
# set the fasta path 
fasta_path = "path/to/g31016_tetraploids_biallelic.fasta"

# empty dictionary for contig sequences - key is contig ID, value is the sequence
# dictionary allows direct lookup by contig name when matching variants in Step 3
fasta_dict = {}

# using SeqIO from Biopython to open the fasta
# context manager (with statement) ensures file is closed safely after reading
with open(fasta_path, "r") as fasta_file:
    for rec in SeqIO.parse(fasta_file, "fasta"):
        # str(rec.seq) converts the Seq object to a plain Python string
        # list(...) converts that string into individual characters e.g. ["A", "T", "G", "C"]
        # stored as a list rather than a string because strings are immutable in Python
        # lists are mutable - individual nucleotides can be swapped in place when applying variants in Step 3
        fasta_dict[rec.id] = list(str(rec.seq)) # store as list of nucleotides

print("Loaded {} contigs from FASTA.".format(len(fasta_dict)))

## Step 3: Apply Variants Based on Allele Frequency

We'll only apply simple SNPs where:
- Top allele frequency > 5%
- It is at least 30% more common than the second allele
- It's a 1-to-1 SNP (not an indel or complex variant)


In [ ]:
# list to store a record of every change made - each entry is a tuple of (contig, position, ref allele, alt allele)
# essential for auditability - can verify exactly what was changed and why
modification_log = []
total_mods = 0  # counter for total variants successfully applied

# iterate over every variant loaded from the VCF
for idx in range(variant_count):
    # unpack the data for this variant from the numpy arrays
    pos = positions[idx]
    ref = ref_alleles[idx]
    raw_alt = alt_alleles[idx]
    chrom = chroms[idx]
    af = allele_freqs[idx]

    # Clean ALT alleles
    # scikit-allel can return ALT alleles as either a comma separated string or an array
    # this handles both cases and produces a clean list either way - defensive programming
    if isinstance(raw_alt, str):
        alt = raw_alt.strip().split(',') # converts a string of nucleotides (eg "T,C,A") to list of strings ["T", "C", "A"]
    else:
        alt = [str(a).strip() for a in raw_alt] # converts anything else to the format above

    # Remove empty alleles
    # splitting on commas can produce empty strings e.g. "T,,C" -> ["T", "", "C"]
    alt = [a for a in alt if a] # get rid of empty strings - if a is False for empty strings
    if not alt:
        continue  # if there's nothing left skip to the next variant

    # Clean AF
    # same defensive pattern as ALT - AF may come back as a single float or an array depending on the variant
    af_values = [af] if isinstance(af, float) else list(af) # if the allele frequency value is a float, put it in a list

    # pair each ALT allele with its frequency then sort highest frequency first
    # zip creates tuples e.g. [("T", 0.8), ("C", 0.15)]
    # lambda x: x[1] tells sorted to rank by the second element of each tuple (the frequency)
    allele_data = sorted(zip(alt, af_values), key=lambda x: x[1], reverse=True) # combine each ALT allele with its AF and sort them in reverse order (lambda x: x[1])
    top_alt, top_af = allele_data[0] # get the most frequent alternate allele and its frequency

    # get second allele frequency for dominance gap calculation
    # defaults to 0.0 if there is no second allele so the filter below doesn't break
    second_af = allele_data[1][1] if len(allele_data) > 1 else 0.0 # if there's a second allele freq else assign 0.0

    # Filters - only allow high quality, clearly dominant SNPs through
    # top allele must exceed 5% frequency - removes rare variants likely to be sequencing noise
    # dominance gap must exceed 30% - ensures the top allele is genuinely dominant, not just marginally ahead
    # e.g. two alleles at 35% and 34% would be arbitrary to apply - filtered out here
    if top_af <= 0.05 or (top_af - second_af) < 0.3: # AF must be > 0.05 and difference between the two AF > 0.30
        continue

    # only allow single nucleotide changes - skip insertions, deletions or complex variants
    # downstream tools (AlphaFold2, HADDOCK) expect clean SNP level changes
    if len(top_alt) != 1 or len(ref) != 1: # if the len of alt or ref allele not equal to 1, skip
        continue

    # Match contig
    # find which contig in the FASTA this variant belongs to
    # and calculate the local offset (index) within that contig's sequence
    contig_id = None  # stays None if no matching contig is found - checked before applying anything
    offset = None
    for fid in fasta_dict:
        if fid.startswith(chrom): # does the contig belong to the same chromosome as this variant
            try:
                # contig IDs are formatted as chr1:1000-2000 - extract the positional range
                region = fid.split(":")[1]
                start, end = map(int, region.split("-"))  # map(int,...) converts both values to integers at once
                if start <= pos <= end:
                    contig_id = fid # does it match
                    offset = pos - start  # convert absolute genomic position to local index within the contig's list
                    break  # match found - stop searching
            except:
                # if contig ID doesn't follow expected format, skip it rather than crashing the pipeline
                # in production you would log these cases rather than silently skipping
                continue

    # Apply change
    # three safety checks before modifying any data - this is the line that actually writes changes
    # contig_id - did we find a matching contig?
    # offset is not None - was an offset successfully calculated?
    # 0 <= offset < len(...) - is the offset within the bounds of the sequence? prevents index out of bounds error
    if contig_id and offset is not None and 0 <= offset < len(fasta_dict[contig_id]):
        fasta_dict[contig_id][offset] = top_alt  # swap the character in the list - this is why sequences are stored as lists
        modification_log.append((contig_id, pos, ref, top_alt))  # log the change as a tuple for auditability
        total_mods += 1  # increment the counter

## Step 4: Save Consensus FASTA and Log of Modifications

The output is:
- A new FASTA file with all accepted SNPs applied
- A text file listing all changes


In [ ]:
# Save new FASTA
output_fasta = "g31016_tetraploids_consensus_AFbased.fasta"
with open(output_fasta, "w") as out_fasta:
    for fid, chars in fasta_dict.items():
        record = SeqRecord(Seq("".join(chars)), id=fid, description="")
        SeqIO.write(record, out_fasta, "fasta")

# Save modification log
with open("modification_log_AFbased.txt", "w") as log_file:
    for contig, pos, ref, alt in modification_log:
        log_file.write("{}\t{}\t{} -> {}\n".format(contig, pos, ref, alt))

print("Done. {} variants applied.".format(total_mods))
